# Notebook nocturna: poda liviana de canaritos + Feature Engineering con algoritmo genético

Pensada para dejar corriendo varias horas sin supervisión. Dos partes:

**Parte 1 -- Poda liviana de canaritos**: en vez de duplicar las 1622 columnas (lo que provocó un error de memoria en una corrida anterior), agrego solo ~30 variables de ruido puro (canaritos), entreno UN modelo, y uso la importancia máxima de esas 30 como umbral de ruido. Cualquier variable real por debajo de ese umbral se descarta. Mucho más liviano en memoria.

**Parte 2 -- Algoritmo genético de FE**: sobre las variables que sobrevivan la poda, busco combinaciones nuevas (`var1 op var2`, con `op` en +,-,*,/) usando un algoritmo evolutivo. El fitness de cada candidata es su AUC univariado contra `clase01` (rápido de calcular, no requiere entrenar un modelo por candidata -- así se pueden evaluar miles de combinaciones por hora). Corre en un loop de tiempo (no de generaciones fijas), con checkpointing constante, así se puede cortar cuando se quiera sin perder el progreso.

**Parte 3 -- Verificación final**: toma las mejores variables encontradas por el genético y las prueba de verdad, agregándolas al modelo base y comparando la ganancia real contra la referencia.

`MIN_DATA_MANUAL` (Parte 3) se fijó en 4000, valor que luego confirmó la grilla de `min_data_in_leaf` del modelo A (`z_grilla_mdl_modeloA`).

## Parte 1 -- Poda liviana de canaritos

In [1]:
rm(list = ls())
gc()

require("data.table")
require("lightgbm")

PARAM_carpeta_datasets <- "C:/Users/Laura/Documents/DMEyF/datasets/"
PARAM_carpeta_exp      <- "C:/Users/Laura/Documents/DMEyF/exp/"

t0 <- Sys.time()
dataset <- fread(paste0(PARAM_carpeta_datasets, "competencia_01_con_lags_fe_historico_rankeado.csv.gz"), stringsAsFactors = TRUE)
cat("Dataset cargado:", nrow(dataset), "filas x", ncol(dataset), "columnas en",
    round(as.numeric(Sys.time()-t0, units="secs"),1), "seg\n")

dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]
campos_buenos <- setdiff(colnames(dataset), c("clase_ternaria", "clase01", "numero_de_cliente"))
campos_buenos <- setdiff(campos_buenos, c("cprestamos_personales", "mprestamos_personales"))

dataset_train <- dataset[foto_mes %in% c(202103, 202104)]
rm(dataset); gc(full = TRUE, verbose = FALSE)

cat(length(campos_buenos), "variables candidatas,", nrow(dataset_train), "filas de train\n")

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,653251,34.9,1447531,77.4,1108991,59.3
Vcells,1199465,9.2,8388608,64.0,1928501,14.8


Caricamento del pacchetto richiesto: data.table

Caricamento del pacchetto richiesto: lightgbm



Dataset cargado: 983061 filas x 1622 columnas en 205.9 seg


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1824280,97.5,2942461,157.2,2942461,157.2
Vcells,443585350,3384.3,2858919992,21811.9,3573528851,27263.9


1618 variables candidatas, 326184 filas de train


### Agrego ~30 canaritos (ruido puro) y entreno UN modelo

In [2]:
set.seed(102191)
N_CANARITOS <- 30

for (i in 1:N_CANARITOS) {
  dataset_train[, paste0("canarito_", i) := runif(.N)]
}

campos_con_canaritos <- c(campos_buenos, paste0("canarito_", 1:N_CANARITOS))

dtrain_canaritos <- lgb.Dataset(
  data = data.matrix(dataset_train[, campos_con_canaritos, with = FALSE]),
  label = dataset_train[, clase01]
)

# uso manual con min_data_in_leaf=4000 (tendencia actual del grid) solo para este chequeo de importancia
param_chequeo <- list(
  boosting= "gbdt", objective= "binary", metric= "auc",
  first_metric_only= FALSE, boost_from_average= TRUE, feature_pre_filter= FALSE,
  force_row_wise= TRUE, verbosity= -100,
  max_depth= -1L, max_bin= 31L, min_sum_hessian_in_leaf = 0.001,
  pos_bagging_fraction= 1.0, neg_bagging_fraction= 1.0, is_unbalance= FALSE,
  drop_rate= 0.1, max_drop= 50, skip_drop= 0.5, extra_trees= FALSE,
  num_iterations = 600, learning_rate = 0.02, feature_fraction = 0.1541961,
  num_leaves = 1684, min_data_in_leaf = 4000, scale_pos_weight = 0.5,
  min_gain_to_split = 0.2158453, lambda_l1 = 1.520327, lambda_l2 = 7.548036,
  bagging_fraction = 0.8425105, bagging_freq = 6, seed = 585581
)

t0 <- Sys.time()
modelo_chequeo <- lgb.train(data = dtrain_canaritos, param = param_chequeo)
cat("Modelo de chequeo entrenado en", round(as.numeric(Sys.time()-t0, units="secs"),1), "seg\n")

imp <- lgb.importance(modelo_chequeo)
setDT(imp)

Modelo de chequeo entrenado en 257.9 seg


### Umbral de ruido y poda

In [3]:
imp_canaritos <- imp[grepl("^canarito_", Feature)]
umbral_ruido <- if (nrow(imp_canaritos) > 0) max(imp_canaritos$Gain) else 0

cat("Umbral de ruido (max Gain entre", N_CANARITOS, "canaritos):", umbral_ruido, "\n")

variables_con_importancia <- imp[!grepl("^canarito_", Feature) & Gain > umbral_ruido, Feature]
# las que ni siquiera aparecen en imp (Gain=0 nunca fueron usadas) tambien se descartan
campos_podados <- intersect(campos_buenos, variables_con_importancia)

cat("Variables originales:", length(campos_buenos), "\n")
cat("Variables que superan el umbral de ruido:", length(campos_podados), "\n")
cat("Variables descartadas:", length(campos_buenos) - length(campos_podados), "\n")

fwrite(data.table(variable = campos_podados), paste0(PARAM_carpeta_exp, "campos_podados_canaritos.txt"))

# libero los canaritos y el modelo de chequeo, ya no los necesito
dataset_train[, (paste0("canarito_", 1:N_CANARITOS)) := NULL]
rm(dtrain_canaritos, modelo_chequeo, imp, imp_canaritos)
gc(full = TRUE, verbose = FALSE)

Umbral de ruido (max Gain entre 30 canaritos): 0.001547273 
Variables originales: 1618 
Variables que superan el umbral de ruido: 110 
Variables descartadas: 1508 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1862828,99.5,4817371,257.3,4817371,257.3
Vcells,443996850,3387.5,1829708796,13959.6,3573528851,27263.9


## Parte 2 -- Algoritmo genético de Feature Engineering

In [4]:
# Solo trabajamos con las variables numéricas podadas (descartamos identificadores si quedaron)
campos_geneticos <- campos_podados[sapply(dataset_train[, ..campos_podados], is.numeric)]
cat(length(campos_geneticos), "variables base para el algoritmo genético\n")

M <- data.matrix(dataset_train[, campos_geneticos, with = FALSE])
y <- dataset_train[, clase01]
n_vars <- ncol(M)

rm(dataset_train); gc(full = TRUE, verbose = FALSE)  # ya no necesito el data.table, solo la matriz M

110 variables base para el algoritmo genético


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1859706,99.4,4817371,257.3,4817371,257.3
Vcells,39357442,300.3,1463767037,11167.7,3573528851,27263.9


### Funciones del algoritmo genético

In [5]:
OPS <- c("+", "-", "*", "/")

auc_rapido <- function(x, y) {
  r <- frank(x, ties.method = "average")
  n1 <- sum(y == 1L); n0 <- length(y) - n1
  if (n1 == 0 || n0 == 0) return(0.5)
  auc <- (sum(r[y == 1L]) - n1 * (n1 + 1) / 2) / (n1 * n0)
  max(auc, 1 - auc)
}

calc_feature <- function(v1, v2, op) {
  out <- switch(op,
    "+" = v1 + v2,
    "-" = v1 - v2,
    "*" = v1 * v2,
    "/" = ifelse(v2 == 0, 0, v1 / v2)
  )
  out[!is.finite(out)] <- 0
  out
}

individuo_random <- function() {
  i1 <- sample.int(n_vars, 1)
  i2 <- sample.int(n_vars, 1)
  while (i2 == i1) i2 <- sample.int(n_vars, 1)
  list(op = sample(OPS, 1), v1 = i1, v2 = i2)
}

fitness_individuo <- function(ind) {
  feat <- calc_feature(M[, ind$v1], M[, ind$v2], ind$op)
  auc_rapido(feat, y)
}

cruza <- function(padreA, padreB) {
  hijo <- list(
    op = if (runif(1) < 0.5) padreA$op else padreB$op,
    v1 = if (runif(1) < 0.5) padreA$v1 else padreB$v1,
    v2 = if (runif(1) < 0.5) padreA$v2 else padreB$v2
  )
  if (hijo$v1 == hijo$v2) hijo$v2 <- sample.int(n_vars, 1)
  hijo
}

mutar <- function(ind, p_mut = 0.15) {
  if (runif(1) < p_mut) ind$op <- sample(OPS, 1)
  if (runif(1) < p_mut) ind$v1 <- sample.int(n_vars, 1)
  if (runif(1) < p_mut) ind$v2 <- sample.int(n_vars, 1)
  if (ind$v1 == ind$v2) ind$v2 <- sample.int(n_vars, 1)
  ind
}

torneo <- function(poblacion, fitnesses, k = 3) {
  idxs <- sample.int(length(poblacion), k)
  mejor <- idxs[which.max(fitnesses[idxs])]
  poblacion[[mejor]]
}

signature <- function(ind) paste(campos_geneticos[ind$v1], ind$op, campos_geneticos[ind$v2])

### Loop evolutivo -- corre por tiempo (no por generaciones fijas), con checkpointing

In [ ]:
MAX_HORAS <- 9          # ajustar según cuánto tiempo se lo deje corriendo
TAM_POBLACION <- 150
N_ELITE <- 15
archivo_top <- paste0(PARAM_carpeta_exp, "genetico_top_features.txt")
archivo_log <- paste0(PARAM_carpeta_exp, "genetico_log.txt")

set.seed(789456)
poblacion <- lapply(1:TAM_POBLACION, function(i) individuo_random())
fitnesses <- sapply(poblacion, fitness_individuo)

top_historico <- data.table(signature = character(), op = character(), var1 = character(),
                             var2 = character(), fitness = numeric())

t_inicio <- Sys.time()
generacion <- 0

repeat {
  generacion <- generacion + 1

  # elitismo: los mejores pasan directo
  orden <- order(-fitnesses)
  nueva_poblacion <- poblacion[orden[1:N_ELITE]]

  # el resto se genera por cruza + mutacion via torneo
  while (length(nueva_poblacion) < TAM_POBLACION) {
    padreA <- torneo(poblacion, fitnesses)
    padreB <- torneo(poblacion, fitnesses)
    hijo <- mutar(cruza(padreA, padreB))
    nueva_poblacion[[length(nueva_poblacion) + 1]] <- hijo
  }

  poblacion <- nueva_poblacion
  fitnesses <- sapply(poblacion, fitness_individuo)

  mejor_gen <- poblacion[[which.max(fitnesses)]]
  mejor_fit <- max(fitnesses)

  # actualizo el top historico (unico por signature, me quedo con las 30 mejores)
  fila <- data.table(signature = signature(mejor_gen), op = mejor_gen$op,
                      var1 = campos_geneticos[mejor_gen$v1], var2 = campos_geneticos[mejor_gen$v2],
                      fitness = mejor_fit)
  top_historico <- rbindlist(list(top_historico, fila))
  top_historico <- unique(top_historico, by = "signature")
  setorder(top_historico, -fitness)
  top_historico <- head(top_historico, 30)

  elapsed_horas <- as.numeric(Sys.time() - t_inicio, units = "hours")

  if (generacion %% 5 == 0) {
    fwrite(top_historico, archivo_top, sep = "\t")
    cat("[", format(Sys.time(), "%X"), "] gen=", generacion, " mejor_fitness=", round(mejor_fit, 5),
        " horas_transcurridas=", round(elapsed_horas, 2), " (checkpoint guardado)\n", sep = "")
    cat(generacion, mejor_fit, round(elapsed_horas,3), "\n", file = archivo_log, append = TRUE)
  }

  if (elapsed_horas >= MAX_HORAS) {
    cat("\n=== TIEMPO LIMITE ALCANZADO (", round(elapsed_horas, 2), " horas,", generacion, " generaciones) ===\n", sep="")
    fwrite(top_historico, archivo_top, sep = "\t")
    break
  }
}

cat("\nTop 10 features encontradas por el genetico:\n")
print(head(top_historico, 10))

## Parte 3 -- Verificación real: ¿las mejores features del genético mejoran la ganancia?

In [ ]:
# Recargo el dataset (se había liberado para el genetico) y materializo las top features encontradas
dataset <- fread(paste0(PARAM_carpeta_datasets, "competencia_01_con_lags_fe_historico_rankeado.csv.gz"), stringsAsFactors = TRUE)
dataset[, clase01 := ifelse(clase_ternaria %in% c("BAJA+1", "BAJA+2"), 1L, 0L)]

top_historico <- fread(paste0(PARAM_carpeta_exp, "genetico_top_features.txt"))
N_FEATURES_NUEVAS <- 10  # cuantas de las mejores agregar al modelo final

top_a_usar <- head(top_historico, N_FEATURES_NUEVAS)
for (i in 1:nrow(top_a_usar)) {
  v1 <- top_a_usar$var1[i]; v2 <- top_a_usar$var2[i]; op <- top_a_usar$op[i]
  nombre <- paste0("gen_", i)
  dataset[, (nombre) := switch(op,
    "+" = get(v1) + get(v2),
    "-" = get(v1) - get(v2),
    "*" = get(v1) * get(v2),
    "/" = ifelse(get(v2) == 0, 0, get(v1) / get(v2))
  )]
}
campos_geneticos_nuevos <- paste0("gen_", 1:nrow(top_a_usar))
cat("Agregadas", length(campos_geneticos_nuevos), "variables nuevas del genetico\n")

campos_podados <- fread(paste0(PARAM_carpeta_exp, "campos_podados_canaritos.txt"))$variable

dataset_train <- dataset[foto_mes %in% c(202103, 202104)]
dfuture       <- dataset[foto_mes == 202106]
clase_ternaria_future <- dfuture[, clase_ternaria]
rm(dataset); gc(full = TRUE, verbose = FALSE)

MIN_DATA_MANUAL <- 4000  # ACTUALIZAR con el ganador real del grid de min_data_in_leaf

param_manual_final <- list(
  boosting= "gbdt", objective= "binary", metric= "auc",
  first_metric_only= FALSE, boost_from_average= TRUE, feature_pre_filter= FALSE,
  force_row_wise= TRUE, verbosity= -100,
  max_depth= -1L, max_bin= 31L, min_sum_hessian_in_leaf = 0.001,
  pos_bagging_fraction= 1.0, neg_bagging_fraction= 1.0, is_unbalance= FALSE,
  drop_rate= 0.1, max_drop= 50, skip_drop= 0.5, extra_trees= FALSE,
  num_iterations = 600, learning_rate = 0.02, feature_fraction = 0.1541961,
  num_leaves = 1684, min_data_in_leaf = MIN_DATA_MANUAL, scale_pos_weight = 0.5,
  min_gain_to_split = 0.2158453, lambda_l1 = 1.520327, lambda_l2 = 7.548036,
  bagging_fraction = 0.8425105, bagging_freq = 6
)

CORTE_FIJO <- 12000
SEMILLAS_3 <- c(585581, 249449, 655649)

evaluar <- function(campos) {
  dtr <- lgb.Dataset(data = data.matrix(dataset_train[, campos, with = FALSE]), label = dataset_train[, clase01])
  mf  <- data.matrix(dfuture[, campos, with = FALSE])
  sapply(SEMILLAS_3, function(SEM) {
    pB <- modifyList(param_manual_final, list(seed = SEM))
    set.seed(SEM, "L'Ecuyer-CMRG")
    modelo <- lgb.train(data = dtr, param = pB)
    pred <- predict(modelo, mf)
    tb <- data.table(clase_ternaria = clase_ternaria_future); tb[, prob := pred]
    setorder(tb, -prob); tb[, Predicted := 0L]; tb[1:CORTE_FIJO, Predicted := 1L]
    tb[Predicted == 1L, sum(ifelse(clase_ternaria == "BAJA+2", 1072500, -27500))]
  })
}

gan_sin_genetico <- evaluar(campos_podados)
gan_con_genetico  <- evaluar(c(campos_podados, campos_geneticos_nuevos))

cat("\n--- SIN variables del genetico (solo podadas) --- \n")
print(gan_sin_genetico); cat("Media:", mean(gan_sin_genetico), "\n\n")
cat("--- CON las", length(campos_geneticos_nuevos), "variables del genetico ---\n")
print(gan_con_genetico); cat("Media:", mean(gan_con_genetico), "\n")